# Inspect saved checkpoints

Read-only companion to `run_summaries.py`. Set `STARFINDER_INSPECTION_ROOT` to its external output directory and start the kernel from `src/python`. This notebook does not run processing. Small saved-formed-v3 development fixtures are not calibrated real-data performance. Physical calibration remains unknown. Live Jupyter requires a separately prepared environment; the code cells also run as ordinary Python.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from starfinder.provenance import read_run
from starfinder.io import load_image_checkpoint, load_candidate_checkpoint, load_final_checkpoint
root = Path(os.environ["STARFINDER_INSPECTION_ROOT"]).resolve()
examples = Path.cwd().parents[1] / "docs/examples"
assert (examples / "checkpoint_inspection.py").is_file(), "Start from src/python"
sys.path.insert(0, str(examples))
from saved_synthetic import verify_files
delivery = json.loads((root / "saved/delivery.json").read_text())
verify_files(root / "saved", delivery)


## True run states and unavailable metrics

A failed run can retain partial round signals. Missing counts are `None`, never zero. A successful run does not certify scientific validity.


In [ ]:
runs = {name: read_run(path) for name, path in {"z9": root/"saved/z9/run", "z1": root/"saved/z1/run", "failed": root/"failed"}.items()}
states = pd.DataFrame([{"case": name, "status": run["status"], "stage_state": run["extensions"]["starfinder.provenance"]["stage_state"], "counts": run["extensions"]["starfinder.provenance"]["final_state"]["counts"]} for name, run in runs.items()])
states


## Images, truth overlays and transforms

Open the generated HTML for channel-specific XY and XZ views, complete histories, and explicit gt-A/gt-B ↔ spot-1/spot-2 correspondence. Display projections do not change stored ZYX coordinates. The helper reuses the accepted W-175 presentation.


In [ ]:
from checkpoint_inspection import inspect_saved
output = root / "notebook-inspection.html"
inspect_saved(root / "saved", output)
print(output)
images = load_image_checkpoint(root/"saved/z9/registered")
[(layer.round_label, layer.loaded.image.shape, layer.loaded.metadata, layer.processing) for layer in images.layers]


## Candidate identities and source traces

Select by namespace and spot ID, never row number or gene assignment. Channels and rounds retain saved order; signals are C×R for one candidate.


In [ ]:
candidates = load_candidate_checkpoint(root/"saved/z9/run/candidates-signals")
final = load_final_checkpoint(root/"saved/z9/run/final-accepted")
identity = {"spot_namespace": candidates.spots.spot_namespace, "spot_id": candidates.intensities.spot_ids[0]}
trace = final.pre_qc.source_trace(**identity)
assert trace["available"]
print(identity, candidates.spots.metadata)
pd.DataFrame(trace["values"], index=candidates.intensities.channel_labels, columns=candidates.intensities.round_labels)


## Complete truth histories, decoded pre-QC and accepted populations

Observed colors, nucleotide decoding, assigned genes and filtering are distinct. The HTML presents saved start-base decoding and independent truth alongside these records. Loss and visibility are retained even for unobserved objects.


In [ ]:
histories = pd.read_parquet(root/"saved/z9/round-truth.parquet")
print(histories.to_string(index=False))
print(final.pre_qc.decoded.table.to_string(index=False))
print(final.filtering.table.to_string(index=False))
print(final.filtering.counts)
final.filtering.accepted


## Access and retention

HTML is standalone and needs no kernel/network. HDF5/Parquet reload additionally needs the saved delivery and source components; original absolute source references must remain reachable. Copying the HTML alone is sufficient for visual review, not raw reload. Do not overwrite earlier reports. Owner Jiahao; retain through thesis/publication; backup/public reproducibility unverified. No MATLAB, scientific or human approval is supplied.
